In [1]:
import torch
from torch import nn

In [11]:
class NiN(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.net = nn.Sequential(
            self.nin_block(output_channels=96, kernel_size=11, stride=4, padding=0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            self.nin_block(output_channels=256, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            self.nin_block(output_channels=384, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout(0.5),
            self.nin_block(output_channels=num_class, kernel_size=3, stride=1, padding=1),
            nn.AdaptiveAvgPool2d(output_size=(1, 1)),
            nn.Flatten()
        )

    def nin_block(self, output_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.LazyConv2d(out_channels=output_channels, kernel_size=kernel_size, stride=stride, padding=padding), nn.ReLU(),
            nn.LazyConv2d(out_channels=output_channels, kernel_size=1), nn.ReLU(),
            nn.LazyConv2d(out_channels=output_channels, kernel_size=1), nn.ReLU()
        )
    def forward(self, X):
        return self.net(X)
    
    def layer_summary(self, X_shape):
        X = torch.randn(X_shape)
        for layer in self.net:
            X = layer(X)
            print(layer.__class__.__name__, 'output shape:\t', X.shape)

In [12]:
model = NiN(10)
model.layer_summary((1, 3, 224, 224))

Sequential output shape:	 torch.Size([1, 96, 54, 54])
MaxPool2d output shape:	 torch.Size([1, 96, 26, 26])
Sequential output shape:	 torch.Size([1, 256, 26, 26])
MaxPool2d output shape:	 torch.Size([1, 256, 12, 12])
Sequential output shape:	 torch.Size([1, 384, 12, 12])
MaxPool2d output shape:	 torch.Size([1, 384, 5, 5])
Dropout output shape:	 torch.Size([1, 384, 5, 5])
Sequential output shape:	 torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 10, 1, 1])
Flatten output shape:	 torch.Size([1, 10])
